# BoTorch Integration Tutorial: Synthetic Test Functions

This tutorial demonstrates how to use BoTorch's qExpectedImprovement (qEI) acquisition function with ALF on synthetic test functions. We'll optimize the Branin function, which has a known global optimum, allowing us to measure convergence objectively.

## Why Synthetic Test Functions?

Synthetic test functions provide several advantages for development and testing:
1. **Known ground truth**: We can measure true regret (distance from optimum)
2. **Fast evaluation**: No expensive experiments needed
3. **Domain agnostic**: Validates that the implementation works for any optimization problem
4. **Standard benchmarks**: Enables comparison with published results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from alf_core import BaseDatasetConfig, Modality
from alf_core.optimizer import Optimizer
from alf_core.oracle import Oracle
from alf_core.surrogate import Surrogate
from alf_core.tasks import DesignTask
from alf_tools.datasets.botorch_test_functions import BoTorchSyntheticDataset
from alf_tools.models.gp import GPModel, GPModelConfig, GPTrainConfig
from alf_tools.optimizer.acquisition_functions.botorch_qei import BoTorchQEI

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Setup: Create the Branin Test Function Dataset

The Branin function is a 2D function with 3 global minima. The known optimal value is -0.397887 (after negation for maximization).

In [ ]:
# Create dataset config
dataset_config = BaseDatasetConfig(
    name="branin",
    modality=Modality.TABULAR,
    seed=42,
    train_ratio=0.05,  # Start with 50 initial samples (5% of 1000)
    validation_frac=0.2,
    test_ratio=0.1,
    split_type="random",
)

# Create Branin dataset
dataset = BoTorchSyntheticDataset(
    config=dataset_config,
    function_name="branin",
    noise_std=0.0,  # Noiseless for this tutorial
    n_initial_samples=1000,
)

dataset.setup()
print(dataset)
print(f"\nTrue optimum: {dataset.true_optimum:.6f}")
print(f"Best in initial training set: {dataset.train_dataset.labels.max():.6f}")

## 2. Configure the Surrogate Model (Gaussian Process)

We'll use a GP with an RBF kernel for the surrogate model.

In [ ]:
# GP model configuration
gp_config = GPModelConfig(
    kernel_type="rbf",
    ard=True,  # Automatic relevance determination
    mean_type="constant",
)

gp_train_config = GPTrainConfig(
    learning_rate=0.01,
    num_iterations=100,
    optimizer_type="adam",
)

# Create GP model
gp_model = GPModel(
    config=gp_config,
    train_config=gp_train_config,
)

# Wrap in Surrogate
surrogate = Surrogate(model=gp_model)

## 3. Configure BoTorch qEI Acquisition Function

BoTorch's qEI jointly optimizes batches of candidates for quality and diversity.

In [ ]:
# Get bounds for optimization
bounds = torch.tensor(dataset.bounds)  # Shape: (2, 2)

# Create BoTorch qEI acquisition function
acquisition_fn = BoTorchQEI(
    batch_size=5,  # Acquire 5 candidates per round
    bounds=bounds,
    num_restarts=10,
    raw_samples=512,
    mc_samples=128,
    seed=42,
)

print(f"Acquisition function: BoTorch qEI with batch_size={acquisition_fn.batch_size}")
print(f"Bounds: {bounds.numpy()}")

## 4. Setup Optimizer and Oracle

For continuous optimization with BoTorch, we use an empty search space (the acquisition function will optimize directly).

In [ ]:
# Create optimizer (no search function needed for continuous optimization)
optimizer = Optimizer(
    acquisition_fn=acquisition_fn,
    search_fn=None,  # qEI will optimize directly
)

# Create oracle (uses dataset.query for evaluation)
oracle = Oracle(scorer=dataset)

## 5. Run Bayesian Optimization

We'll run 20 rounds of acquisition, acquiring 5 candidates per round (100 total evaluations).

In [ ]:
# Create design task
task = DesignTask(
    num_acq_rounds=20,
    acq_batch_size=5,
)

# Setup task
state = task.setup(
    dataset=dataset,
    surrogate=surrogate,
)

# Run optimization
results = task.run(
    state=state,
    optimizer=optimizer,
    oracle=oracle,
)

print("\nOptimization complete!")
print(f"Best value found: {results.best_value:.6f}")
print(f"True optimum: {dataset.true_optimum:.6f}")
print(f"Final regret: {dataset.true_optimum - results.best_value:.6f}")

## 6. Analyze Results

Let's visualize the optimization progress with regret curves.

In [ ]:
# Extract metrics
rounds = []
best_values = []
regrets = []

for round_num, round_results in results.acquisition_rounds.items():
    rounds.append(round_num)
    best_val = round_results["best_train_value"]
    best_values.append(best_val)
    regrets.append(dataset.true_optimum - best_val)

# Plot convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Best value over rounds
ax1.plot(rounds, best_values, marker="o", linewidth=2, markersize=6)
ax1.axhline(y=dataset.true_optimum, color="r", linestyle="--", label="True optimum")
ax1.set_xlabel("Acquisition Round", fontsize=12)
ax1.set_ylabel("Best Value Found", fontsize=12)
ax1.set_title("Optimization Progress", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Regret over rounds (log scale)
ax2.semilogy(rounds, regrets, marker="o", linewidth=2, markersize=6, color="orange")
ax2.set_xlabel("Acquisition Round", fontsize=12)
ax2.set_ylabel("Regret (log scale)", fontsize=12)
ax2.set_title("Regret Convergence", fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal regret: {regrets[-1]:.6f}")
print(f"Improvement from initial: {(best_values[-1] - best_values[0]):.6f}")

## 7. Comparison with Greedy Expected Improvement (Optional)

Let's compare BoTorch's qEI with a standard greedy EI approach.

In [ ]:
from alf_tools.optimizer.acquisition_functions.expected_improvement import ExpectedImprovement
from alf_tools.optimizer.search.single_mutant_search import DatasetSearch

# Reset dataset
dataset_greedy = BoTorchSyntheticDataset(
    config=dataset_config,
    function_name="branin",
    noise_std=0.0,
    n_initial_samples=1000,
)
dataset_greedy.setup()

# Create greedy optimizer
greedy_acq = ExpectedImprovement()
search_fn = DatasetSearch()  # Searches candidate pool
optimizer_greedy = Optimizer(acquisition_fn=greedy_acq, search_fn=search_fn)
oracle_greedy = Oracle(scorer=dataset_greedy)

# Run greedy optimization
task_greedy = DesignTask(num_acq_rounds=20, acq_batch_size=5)
state_greedy = task_greedy.setup(
    dataset=dataset_greedy,
    surrogate=Surrogate(model=GPModel(config=gp_config, train_config=gp_train_config)),
)
results_greedy = task_greedy.run(
    state=state_greedy, optimizer=optimizer_greedy, oracle=oracle_greedy
)

print(f"\nGreedy EI final best value: {results_greedy.best_value:.6f}")
print(f"BoTorch qEI final best value: {results.best_value:.6f}")
print(f"Improvement of qEI over greedy: {(results.best_value - results_greedy.best_value):.6f}")

## Summary

In this tutorial, we:
1. Created a synthetic Branin dataset using BoTorch test functions
2. Configured a Gaussian Process surrogate model
3. Used BoTorch's qEI for batch acquisition
4. Ran Bayesian optimization and measured convergence to the known optimum
5. Compared with greedy Expected Improvement

**Key Takeaways:**
- BoTorch's qEI jointly optimizes batches for quality and diversity
- Synthetic test functions enable rapid validation with known ground truth
- The same code works for any continuous optimization problem

**Next Steps:**
- Try other test functions (Hartmann, Ackley) to test on different landscapes
- Experiment with different batch sizes and MC samples
- Apply to real biological design problems (proteins, molecules)